# Import Libraries and Define Project Folders

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from keplergl import KeplerGl
import json

PROJECT_ROOT = Path("..")
DATA_DIR = PROJECT_ROOT / "data"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
DATA_DIR.mkdir(exist_ok=True)
OUTPUTS_DIR.mkdir(exist_ok=True)

plt.rcParams["figure.dpi"] = 120

/opt/anaconda3/envs/citibike-dashboard/lib/python3.11/site-packages/keplergl/keplergl.py:13: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_string


# Load Data

In [30]:
df = pd.read_csv(
    DATA_DIR / "cbsd_main_2022_02.csv",
    dtype={"start_station_id": "string", "end_station_id": "string"},
    low_memory=False
)

# parse dates safely (errors='coerce' prevents crashes if any weird rows exist)
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df["started_at"] = pd.to_datetime(df["started_at"], errors="coerce")
df["ended_at"] = pd.to_datetime(df["ended_at"], errors="coerce")

df.shape

(29838166, 15)

# Data Wrangling for creating the Kepler.gl map

In [31]:
# create a new column 
df["value"] = 1

# create new df_group with the columns needed for the map - including the new column for trips count
df_group = (
    df.groupby(["start_station_name", "end_station_name"])["value"]
      .count()
      .reset_index()
      .rename(columns={"value": "trips"})
)

coords = (
    df.groupby(["start_station_name", "end_station_name"])
      .agg(
          start_lat=("start_lat", "first"),
          start_lng=("start_lng", "first"),
          end_lat=("end_lat", "first"),
          end_lng=("end_lng", "first"),
      )
      .reset_index()
)

df_final = df_group.merge(coords, on=["start_station_name", "end_station_name"], how="left")

df_final.head()

,start_station_name,end_station_name,trips,start_lat,start_lng,end_lat,end_lng
0,1 Ave & E 110 St,1 Ave & E 110 St,791,40.792327,-73.9383,40.792327,-73.938300
1,1 Ave & E 110 St,1 Ave & E 18 St,2,40.792327,-73.9383,40.733812,-73.980544
2,1 Ave & E 110 St,1 Ave & E 30 St,4,40.792327,-73.9383,40.741444,-73.975361
3,1 Ave & E 110 St,1 Ave & E 39 St,1,40.792327,-73.9383,40.747140,-73.971130
4,1 Ave & E 110 St,1 Ave & E 44 St,12,40.792327,-73.9383,40.750020,-73.969053


In [41]:
df_final.shape

(1013397, 7)

In [4]:
# create station-level summary for mapping station demand
station_summary = (
    df.groupby("start_station_name")
      .agg(
          trips=("start_station_name", "size"),
          start_lat=("start_lat", "median"),
          start_lng=("start_lng", "median"),
      )
      .reset_index()
      .dropna(subset=["start_station_name", "start_lat", "start_lng"])
      .sort_values("trips", ascending=False)
      .reset_index(drop=True)
)

station_summary.head()

,start_station_name,trips,start_lat,start_lng
0,W 21 St & 6 Ave,129016,40.741740,-73.994156
1,West St & Chambers St,123289,40.717548,-74.013221
2,Broadway & W 58 St,114293,40.766953,-73.981693
3,6 Ave & W 33 St,106440,40.749013,-73.988484
4,1 Ave & E 68 St,104856,40.765005,-73.958185


# Add Station Rank and Group Top 50% and Bottom 50% of stations

In [5]:
station_summary["station_rank"] = range(1, len(station_summary) + 1)

cutoff = int(np.ceil(len(station_summary) * 0.50))

station_summary["station_group"] = np.where(
    station_summary["station_rank"] <= cutoff,
    "Top 50%",
    "Bottom 50%"
)

station_summary["station_group"].value_counts()

station_group
Top 50%       881
Bottom 50%    880
Name: count, dtype: int64

# Create the Kepler.gl map for Stations

In [12]:
with open(OUTPUTS_DIR / "cbsd_kepler_stations_config.json", "r") as f:
    stations_config = json.load(f)

m_stations = KeplerGl(height=700, config=stations_config, show_docs=False)
m_stations.add_data(data=station_summary, name="citi_bike_stations")
m_stations

KeplerGl(config={'version': 'v1', 'config': {'visState': {'filters': [], 'layers': [{'id': '8na287h', 'type': …

# Save customized output

In [18]:
config = m.config
with open(OUTPUTS_DIR / "cbsd_kepler_stations_config.json", "w") as f:
    json.dump(config, f)

m.save_to_html(
    file_name=str(OUTPUTS_DIR / "cbsd_kepler_stations_configured.html"),
    read_only=False,
    config=config
)

Map saved to ../outputs/cbsd_kepler_stations_configured.html!


# Top 20 Stations

In [14]:
with open(OUTPUTS_DIR / "cbsd_kepler_top20_config.json", "r") as f:
    top20_config = json.load(f)

m_top20 = KeplerGl(height=700, config=top20_config, show_docs=False)
m_top20.add_data(data=top20_stations_map, name="top20_stations")
m_top20

KeplerGl(config={'version': 'v1', 'config': {'visState': {'filters': [], 'layers': [{'id': 'va30a9u', 'type': …

In [17]:
config = m_top20.config
with open(OUTPUTS_DIR / "cbsd_kepler_top20_config.json", "w") as f:
    json.dump(config, f)

m_top20.save_to_html(
    file_name=str(OUTPUTS_DIR / "cbsd_kepler_top20_configured.html"),
    read_only=False,
    config=config
)

Map saved to ../outputs/cbsd_kepler_top20_configured.html!


# Most repeated trip corridors

In [44]:
trips_map_df = df_final.nlargest(100, "trips").copy()

In [45]:
# Create top 100 most repeated aggregated trip corridors
trips_map_df = df_final.nlargest(100, "trips").copy()

# Build a fresh Kepler map from that smaller dataframe
m_trips_top100 = KeplerGl(
    height=700,
    data={"citi_bike_trips": trips_map_df},
    show_docs=False
)

m_trips_top100

KeplerGl(data={'citi_bike_trips':                        start_station_name                   end_station_name…

In [46]:
config = m_trips_top100.config

with open(OUTPUTS_DIR / "cbsd_kepler_trips_top100_config.json", "w") as f:
    json.dump(config, f)

m_trips_top100.save_to_html(
    file_name=str(OUTPUTS_DIR / "cbsd_kepler_trips_top100_configured.html"),
    read_only=False,
    config=config
)

Map saved to ../outputs/cbsd_kepler_trips_top100_configured.html!
